# How to use vLLM as a LLM service

## Step 1: Launch vLLM Server
With the container ready, the next step is launching the vLLM server. Please **open a new terminal** to run following script:

```bash
export TP=1
export MODEL_ID="Qwen/Qwen3-8B"
export VLLM_ROCM_USE_AITER=0
export VLLM_USE_AITER_UNIFIED_ATTENTION=0
export VLLM_ROCM_USE_AITER_MHA=0

vllm serve \
    $MODEL_ID \
    --port 8000 \
    --tensor-parallel $TP \
    --no-enable-prefix-caching \
    --disable-log-requests \
    --compilation-config '{"cudagraph_mode": "FULL"}' 
```

On AMD Radeon™ PRO W7900D, we use vLLM's built-in attention backend and Qwen3-8B in BF16, since the upstream AITER and GPT-OSS MXFP4 kernels do not support this GPU. We then start the Qwen3-8B model using the vllm serve command. Key options include `--tensor-parallel` to define GPU count, and `--compilation-config` to manage graph compilation and sequence length. Prefix caching and logging can be toggled to cut overhead. Once launched, your GPU runs a production-ready inference server, essentially turning it into a private mini-ChatGPT.


After a few minutes, if you have logs as below, then congrats, vLLM server is successfully on now!
```text
INFO:     Started server process [1022120]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
```

## Step 2: Send a request to the launched server
To test the vLLM server, send an HTTP request using curl. Specify the model, a prompt like `The future of AI is` and parameters such as `max_tokens` and `temperature`. The server processes the request on the GPU and returns generated text. This confirms the server is operational and optimized, providing a functional inference API for integration into applications.

In [ ]:
!curl http://localhost:8000/v1/completions \
    -H "Content-Type: application/json" \
    -d '{  \
        "model": "Qwen/Qwen3-8B",  \
        "prompt": "The future of AI is",  \
        "max_tokens": 100,  \
        "temperature": 0  \
    }'  

# How to Benchmark vLLM Performance



## vllm bench serve 
This command tests online serving performance under concurrent loads (e.g., 32 users) using synthetic random data. It measures real-world metrics like Time to First Token (TTFT), Time Per Output Token (TPOT), Inter-Token Latency (ITL), and End-to-End Latency (E2EL) to evaluate responsiveness and stability in API-like environments. Before running command below, you should firstly launch vLLM server following [Launch vLLM Server](#step-1-launch-vllm-server)

In [ ]:
MODEL_ID = "Qwen/Qwen3-8B"
CONTAINER_NAME="vllm-demo"

!vllm bench serve \
    --model {MODEL_ID} \
    --dataset-name random \
    --dataset-name random \
    --random-input-len 4096 \
    --random-output-len 1024 \
    --max-concurrency 8 \
    --num-prompts 80 \
    --ignore-eos \
    --percentile_metrics ttft,tpot,itl,e2el

`--max-concurrency`: This defines the maximum number of requests that can be actively processed by the server at the same time. It acts like a gatekeeper, controlling the level of parallel load on the system. 

`--num-prompts`: This is the total number of requests (prompts) that the benchmark will send. 

The recommendation to set num-prompts to 10 times the max-concurrency value is a best practice for achieving statistically reliable and stable performance metrics during a benchmark test. It’s better to set num-prompts as 10 times of max-concurrency value. The general setting of `--max-concurrency` for MI300X is [128 64 32 16 8]. 

## vllm bench latency

Since `vllm bench latency` and `vllm bench throughput` will launch a new vllm service for benchmark, before we advance, please run following command to kill running vLLM service.

In [ ]:
!pkill -9 -f VLLM

 
Evaluates single-request performance without parallelism (`tensor-parallel=1`). It measures prompt processing and token generation latency for individual requests, highlighting baseline performance without concurrency or batching effects, ideal for low-load scenarios.

In [ ]:
MODEL_ID = "Qwen/Qwen3-8B"
CONTAINER_NAME="vllm-demo"

!vllm bench latency \
    --model {MODEL_ID} \
    --input-len 4096 \
    --output-len 1024 \
    --tensor-parallel 1 

## vllm bench throughput

Since `vllm bench latency` and `vllm bench throughput` will launch a new vllm service for benchmark, before we advance, please run following command to kill running vLLM service.

In [ ]:
!pkill -9 -f VLLM

Focuses on maximizing token generation speed under hardware optimization (e.g., 4-way tensor parallelism). It stresses batch processing efficiency by generating fixed-length outputs, measuring raw tokens/second, and ignoring concurrency dynamics to benchmark peak computational throughput.

Batch processing: Batch processing groups multiple requests together to be processed simultaneously by the AI. This maximizes hardware efficiency and increases overall throughput, but it can increase the latency for any individual request. It is ideal for non-interactive, high-volume tasks. 

`--tensor-parallel n` which `n` value set the card number we adopt to serve the model.

In [ ]:
MODEL_ID = "Qwen/Qwen3-8B"
CONTAINER_NAME="vllm-demo"

!vllm bench throughput \
    --model {MODEL_ID} \
    --dataset-name random \
    --input-len 4096 \
    --output-len 1024 \
    --num-prompts 4 \
    --tensor-parallel 1 